# Random Forest Model

In [ ]:
"""
RandomForest hyperparameter sweep (multiclass).

What it does
------------
1) Sweeps over a manual RF param grid.
2) For EACH hyperparameter combo:
   - trains on train split
   - evaluates metrics on val + test
   - appends ONE ROW immediately to a CSV (append mode) => crash-safe
3) Resume capability:
   - if CSV exists, it skips param combos already completed (using stable param_id)
4) After sweep:
   - selects best hyperparams by SELECT_BEST_BY (default val_macro_f1)
   - retrains best model and saves ONLY that model + metadata

Outputs
-------
- results_dir/
    - sweep_metrics.csv                 (incrementally appended; resume-safe)
    - rf_sweep_log_<timestamp>.txt
    - rf_best_model_<timestamp>/
        - random_forest.joblib
        - metadata_rf.json
        - test_outputs.npz
"""

import os
import sys
import json
import time
import hashlib
import random
import itertools
import contextlib
from datetime import datetime

import torch
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight
import joblib


# =====================================================
# USER CONFIG
# =====================================================
DATA_PATH = "../Data/multiclass_data_no_SMOTE"
RESULTS_DIR = "../Results/results_rf_hparam_sweep"

USE_BALANCED_WEIGHTS = False
GLOBAL_SEED = 42

# Options (must exist as a CSV column):
# "val_macro_f1", "val_bal_acc", "val_macro_auc_ovr", "val_mcc"
SELECT_BEST_BY = "val_macro_f1"

# Class names for metadata only (optional)
CLASS_ID_TO_NAME = {
    0: "phosphate",
    1: "sulfate",
    2: "chloride",
    3: "nitrate",
    4: "carbonate",
}

# =====================================================
# PARAM GRID
# =====================================================
PARAM_GRID = {
    "n_estimators": [200],
    "max_depth": [None],
    "min_samples_split": [2],
    "min_samples_leaf": [1],
    "criterion": ["gini"],
    "max_features": [0.2],  
    "bootstrap": [False],
}


# =====================================================
# REPRODUCIBILITY
# =====================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(GLOBAL_SEED)


# =====================================================
# IO SETUP
# =====================================================
os.makedirs(RESULTS_DIR, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d-%H%M%S")
LOG_PATH = os.path.join(RESULTS_DIR, f"rf_sweep_log_{ts}.txt")

# Stable CSV path for resume
CSV_PATH = os.path.join(RESULTS_DIR, "sweep_metrics.csv")

BEST_MODEL_DIR = os.path.join(RESULTS_DIR, f"rf_best_model_{ts}")
os.makedirs(BEST_MODEL_DIR, exist_ok=True)


# =====================================================
# TEE LOGGER
# =====================================================
class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
    def flush(self):
        for s in self.streams:
            try:
                s.flush()
            except Exception:
                pass


# =====================================================
# LOAD DATA
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device (RF runs on CPU; this is only for loading tensors).")

print("📦 Loading datasets...")
train = torch.load(os.path.join(DATA_PATH, "training_data.pt"), map_location="cpu")
val   = torch.load(os.path.join(DATA_PATH, "validation_data.pt"), map_location="cpu")
test  = torch.load(os.path.join(DATA_PATH, "test_data.pt"), map_location="cpu")
print("✅ Data loaded successfully!")

def to_np(t):
    return t.detach().cpu().numpy() if torch.is_tensor(t) else np.array(t)

X_train, y_train = to_np(train["X_train"]), to_np(train["y_train"]).ravel().astype(int)
X_val,   y_val   = to_np(val["X_val"]),     to_np(val["y_val"]).ravel().astype(int)
X_test,  y_test  = to_np(test["X_test"]),   to_np(test["y_test"]).ravel().astype(int)

classes = np.unique(y_train)
n_classes = len(classes)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"Classes: {n_classes} | Train distribution: {np.bincount(y_train)}")
print(f"CSV (incremental): {CSV_PATH}")


# =====================================================
# CLASS WEIGHTS
# =====================================================
if USE_BALANCED_WEIGHTS:
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, cw)}
    print("Using balanced class weights:", class_weight_dict)
else:
    class_weight_dict = None
    print("Using uniform weights (no balancing)")

# =====================================================
# PARAM UTILITIES
# =====================================================
def stable_param_id(params: dict) -> str:
    s = json.dumps(params, sort_keys=True, default=str)
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]

def load_completed_param_ids(csv_path: str) -> set:
    if not os.path.exists(csv_path):
        return set()
    try:
        df = pd.read_csv(csv_path)
        if "param_id" not in df.columns:
            return set()
        return set(df["param_id"].astype(str).tolist())
    except Exception:
        return set()

# =====================================================
# ROC-AUC HELPERS
# =====================================================
def safe_auc(y_true, y_proba, average="macro"):
    try:
        return roc_auc_score(y_true, y_proba, multi_class="ovr", average=average)
    except Exception:
        return np.nan

def save_prediction_outputs(save_dir, y_true, y_pred, y_proba, class_names=None, prefix="test"):
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, f"{prefix}_outputs.npz")

    if class_names is None:
        class_names = np.array([f"C{i}" for i in range(y_proba.shape[1])], dtype=object)
    else:
        class_names = np.array(class_names, dtype=object)

    np.savez_compressed(
        save_path,
        y_true=np.asarray(y_true),
        y_pred=np.asarray(y_pred),
        y_proba=np.asarray(y_proba),
        class_names=class_names
    )
    print(f"Saved prediction outputs to: {save_path}")

# =====================================================
# METRICS
# =====================================================
def per_class_f1_dict(y_true, y_pred, n_classes):
    scores = f1_score(y_true, y_pred, average=None, labels=np.arange(n_classes))
    return {f"f1_c{i}": float(scores[i]) for i in range(n_classes)}

def compute_metrics_with_per_class(y_true, y_pred, y_proba, n_classes):
    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["bal_acc"] = float(balanced_accuracy_score(y_true, y_pred))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))
    out["weighted_f1"] = float(f1_score(y_true, y_pred, average="weighted"))
    out["micro_f1"] = float(f1_score(y_true, y_pred, average="micro"))
    out["mcc"] = float(matthews_corrcoef(y_true, y_pred))
    out["macro_auc_ovr"] = float(safe_auc(y_true, y_proba, average="macro"))
    out["weighted_auc_ovr"] = float(safe_auc(y_true, y_proba, average="weighted"))
    out.update(per_class_f1_dict(y_true, y_pred, n_classes))
    return out


# =====================================================
# CSV APPEND (CRASH-SAFE)
# =====================================================
def append_row_to_csv(csv_path: str, row: dict):
    row_df = pd.DataFrame([row])
    if not os.path.exists(csv_path):
        row_df.to_csv(csv_path, index=False)
    else:
        row_df.to_csv(csv_path, mode="a", header=False, index=False)


# =====================================================
# BUILD PARAM COMBOS
# =====================================================
keys = list(PARAM_GRID.keys())
values = list(PARAM_GRID.values())
param_combinations = [dict(zip(keys, combo)) for combo in itertools.product(*values)]
print(f"Total combos in grid: {len(param_combinations)}")


# =====================================================
# MAIN SWEEP (RESUME-CAPABLE)
# =====================================================
log_f = open(LOG_PATH, "w", encoding="utf-8")
tee = Tee(sys.stdout, log_f)

try:
    with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
        completed = load_completed_param_ids(CSV_PATH)
        print(f"Resume: found {len(completed)} completed combos in existing CSV.")

        t0 = time.time()

        for idx, params in enumerate(param_combinations, 1):
            # ✅ FIX: ensure correct dtype for max_features
            if isinstance(params["max_features"], str) and params["max_features"] not in ["sqrt", "log2"]:
                params["max_features"] = float(params["max_features"])
            pid = stable_param_id(params)
            if pid in completed:
                print(f"[{idx}/{len(param_combinations)}] SKIP (done) param_id={pid} params={params}")
                continue

            print(f"\n[{idx}/{len(param_combinations)}] RUN param_id={pid} params={params}")

            row = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "param_id": pid,
                "use_balanced_weights": bool(USE_BALANCED_WEIGHTS),
                "seed": int(GLOBAL_SEED),
                **params
            }

            try:
                model = RandomForestClassifier(
                    random_state=GLOBAL_SEED,
                    n_jobs=-1,
                    class_weight=class_weight_dict,
                    **params
                )
                model.fit(X_train, y_train)

                # VAL
                y_val_pred = model.predict(X_val)
                y_val_proba = model.predict_proba(X_val)
                val_metrics = compute_metrics_with_per_class(y_val, y_val_pred, y_val_proba, n_classes)

                # TEST
                y_test_pred = model.predict(X_test)
                y_test_proba = model.predict_proba(X_test)
                test_metrics = compute_metrics_with_per_class(y_test, y_test_pred, y_test_proba, n_classes)

                row.update({f"val_{k}": v for k, v in val_metrics.items()})
                row.update({f"test_{k}": v for k, v in test_metrics.items()})

                append_row_to_csv(CSV_PATH, row)
                completed.add(pid)

                print(
                    f"VAL  acc={row['val_acc']:.4f} bal_acc={row['val_bal_acc']:.4f} "
                    f"macro_f1={row['val_macro_f1']:.4f} mcc={row['val_mcc']:.4f} "
                    f"macro_auc={row['val_macro_auc_ovr']:.4f}"
                )
                print(
                    f"TEST acc={row['test_acc']:.4f} bal_acc={row['test_bal_acc']:.4f} "
                    f"macro_f1={row['test_macro_f1']:.4f} mcc={row['test_mcc']:.4f} "
                    f"macro_auc={row['test_macro_auc_ovr']:.4f}"
                )

            except Exception as e:
                # Append an error row too, so you know it failed.
                row["error"] = str(e)
                append_row_to_csv(CSV_PATH, row)
                completed.add(pid)
                print(f"❌ FAILED param_id={pid} error={e}")

        print(f"\nSweep finished. Wall time: {time.time()-t0:.1f} sec")
        print(f"CSV saved incrementally at: {CSV_PATH}")

        # =====================================================
        # SELECT BEST + TRAIN AGAIN + SAVE ONLY BEST MODEL
        # =====================================================
        df = pd.read_csv(CSV_PATH)

        # Only successful runs
        if "error" in df.columns:
            df_ok = df[df["error"].isna()]
        else:
            df_ok = df

        if df_ok.empty:
            raise RuntimeError("No successful runs found in CSV; cannot select best model.")

        if SELECT_BEST_BY not in df_ok.columns:
            raise RuntimeError(f"SELECT_BEST_BY='{SELECT_BEST_BY}' not found in CSV columns.")

        df_ok = df_ok.sort_values(SELECT_BEST_BY, ascending=False)
        best_row = df_ok.iloc[0].to_dict()

        best_params = {k: best_row[k] for k in keys}

        # normalize types for sklearn
        best_params["n_estimators"] = int(best_params["n_estimators"])
        best_params["min_samples_split"] = int(best_params["min_samples_split"])
        best_params["min_samples_leaf"] = int(best_params["min_samples_leaf"])
        # max_depth may be NaN in CSV when None; handle it:
        if pd.isna(best_params["max_depth"]):
            best_params["max_depth"] = None
        else:
            best_params["max_depth"] = int(best_params["max_depth"])
        # max_features can be string or float
        if isinstance(best_params["max_features"], str) and best_params["max_features"] not in ["sqrt", "log2"]:
            best_params["max_features"] = float(best_params["max_features"])
        else:
            best_params["max_features"] = float(best_params["max_features"])
        best_params["bootstrap"] = bool(best_params["bootstrap"])

        print("\n" + "=" * 90)
        print(f"BEST PARAMS by {SELECT_BEST_BY} = {best_row[SELECT_BEST_BY]:.6f}")
        print(best_params)
        print("=" * 90)

        print("\nTraining best RF again (for saving)...")
        final_model = RandomForestClassifier(
            random_state=GLOBAL_SEED,
            n_jobs=-1,
            class_weight=class_weight_dict,
            **best_params
        )
        final_model.fit(X_train, y_train)

        model_path = os.path.join(BEST_MODEL_DIR, "random_forest.joblib")
        joblib.dump(final_model, model_path)

        # =====================================================
        # Save prediction outputs for confusion matrix / ROC
        # =====================================================
        y_pred = final_model.predict(X_test)
        y_proba = final_model.predict_proba(X_test)

        save_prediction_outputs(
            save_dir=BEST_MODEL_DIR,
            y_true=y_test,
            y_pred=y_pred,
            y_proba=y_proba,
            class_names=[CLASS_ID_TO_NAME[i] for i in range(n_classes)],
            prefix="test"
        )

        meta = {
            "data_path": DATA_PATH,
            "results_csv": CSV_PATH,
            "select_best_by": SELECT_BEST_BY,
            "best_score": float(best_row[SELECT_BEST_BY]),
            "best_param_id": str(best_row["param_id"]),
            "best_params": best_params,
            "use_balanced_weights": bool(USE_BALANCED_WEIGHTS),
            "seed": int(GLOBAL_SEED),
            "feature_dim": int(X_train.shape[1]),
            "n_classes": int(n_classes),
            "class_id_to_name": {str(k): v for k, v in CLASS_ID_TO_NAME.items()},
            "timestamp_saved": ts,
        }
        meta_path = os.path.join(BEST_MODEL_DIR, "metadata_rf.json")
        with open(meta_path, "w", encoding="utf-8") as jf:
            json.dump(meta, jf, indent=2)

        print(f"\n✅ Saved BEST model: {model_path}")
        print(f"✅ Saved BEST metadata: {meta_path}")

finally:
    log_f.close()
    print("Log file closed.")

print("\nDone.")

Using cuda device (RF runs on CPU; this is only for loading tensors).
📦 Loading datasets...
✅ Data loaded successfully!
Train: X=(169692, 1280), y=(169692,)
Val:   X=(21211, 1280), y=(21211,)
Test:  X=(21212, 1280), y=(21212,)
Classes: 5 | Train distribution: [33938 33939 33938 33938 33939]
CSV (incremental): training_runs/smote_study/results_rf_best_hparam_smote/sweep_metrics.csv
Using uniform weights (no balancing)
Total combos in grid: 12
Resume: found 11 completed combos in existing CSV.
[1/12] SKIP (done) param_id=171f36f06824 params={'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'criterion': 'gini', 'max_features': 0.2, 'bootstrap': False}
[2/12] SKIP (done) param_id=ae75a831a2c3 params={'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'criterion': 'gini', 'max_features': 0.5, 'bootstrap': False}
[3/12] SKIP (done) param_id=a47bef876928 params={'n_estimators': 200, 'max_depth': None, 'min_samples_split

In [ ]:
# =====================================================
# 📊 CONFUSION MATRIX (FULLY CUSTOMIZABLE - SINGLE CELL)
# =====================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# =========================
# 🔧 USER INPUT
# =========================
NPZ_PATH = "../Results/results_rf_hparam_sweep/best_model/test_outputs.npz"   
SAVE_PATH = "../Results/results_rf_hparam_sweep/best_model/confusion_matrix.png"

# =========================
# 🎨 STYLE OPTIONS
# =========================
FONT_FAMILY = "DejaVu Sans"     
TITLE_SIZE = 16
LABEL_SIZE = 12
TICK_SIZE = 12
ANNOT_SIZE = 12

FIGSIZE = (6, 5)
DPI = 300

CMAP = "Blues"            # try: "viridis", "magma", "coolwarm"
SHOW_VALUES = True
FORMAT = ".2f"

NORMALIZE = True          # True = normalized, False = raw counts

# Grid / borders
SHOW_GRID = False
LINEWIDTH = 0.4
LINECOLOR = "black"

# Colorbar
SHOW_CBAR = True
CBAR_LABEL = ""

# Tick rotation
XTICK_ROT = 45
YTICK_ROT = 0

# =========================
# 📂 LOAD DATA
# =========================
data = np.load(NPZ_PATH, allow_pickle=True)

y_true = data["y_true"]
y_pred = data["y_pred"]
class_names = data["class_names"]

# =========================
# 📊 CONFUSION MATRIX
# =========================
cm = confusion_matrix(y_true, y_pred)

if NORMALIZE:
    cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm = np.nan_to_num(cm)
    title = "Normalized Confusion Matrix"
    vmin, vmax = 0.0, 1.0
else:
    title = "Confusion Matrix"
    FORMAT = "d"
    vmin, vmax = None, None

# =========================
# 🎨 PLOT SETTINGS
# =========================
plt.rcParams["font.family"] = FONT_FAMILY

plt.figure(figsize=FIGSIZE, dpi=DPI)

ax = sns.heatmap(
    cm,
    annot=SHOW_VALUES,
    fmt=FORMAT,
    cmap=CMAP,
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=SHOW_CBAR,
    linewidths=LINEWIDTH if SHOW_GRID else 0,
    linecolor=LINECOLOR,
    vmin=vmin,
    vmax=vmax,
    annot_kws={"size": ANNOT_SIZE}
)

# Labels and title
plt.xlabel("Predicted Label", fontsize=LABEL_SIZE)
plt.ylabel("True Label", fontsize=LABEL_SIZE)
plt.title(title, fontsize=TITLE_SIZE)

# Tick formatting
plt.xticks(rotation=XTICK_ROT, ha="right", fontsize=TICK_SIZE)
plt.yticks(rotation=YTICK_ROT, fontsize=TICK_SIZE)

# Colorbar label
if SHOW_CBAR:
    cbar = ax.collections[0].colorbar
    cbar.set_label(CBAR_LABEL, fontsize=LABEL_SIZE)

plt.tight_layout()

# =========================
# 💾 SAVE (optional)
# =========================
if SAVE_PATH:
    plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches="tight")
    print(f"Saved figure → {SAVE_PATH}")

plt.show()

In [ ]:
# =====================================================
# ROC CURVE: per-class dotted lines + macro mean black
# with bootstrap std-dev shaded band
# =====================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# =========================
# USER INPUT
# =========================
NPZ_PATH = "../Results/results_rf_hparam_sweep/best_model/test_outputs.npz"
SAVE_PATH = "../Results/results_rf_hparam_sweep/best_model/roc_auc.png"

N_BOOTSTRAPS = 300  
RANDOM_SEED = 42

# =========================
# STYLE OPTIONS
# =========================
FONT_FAMILY = "DejaVu Sans"
FIGSIZE = (7, 6)
DPI = 600

TITLE_SIZE = 18
LABEL_SIZE = 16
TICK_SIZE = 14
LEGEND_SIZE = 10

MACRO_LINEWIDTH = 1.5
CLASS_LINEWIDTH = 1.5
STD_ALPHA = 0.25

MACRO_COLOR = "black"
STD_COLOR = "gray"
CLASS_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

# =========================
# LOAD DATA
# =========================
data = np.load(NPZ_PATH, allow_pickle=True)
y_true = data["y_true"]
y_proba = data["y_proba"]
class_names = data["class_names"]

n_classes = y_proba.shape[1]

# =========================
# BINARIZE LABELS
# =========================
y_bin = label_binarize(y_true, classes=np.arange(n_classes))

# =========================
# HELPERS
# =========================
def compute_per_class_roc(y_bin_local, y_score_local):
    fpr_dict, tpr_dict, auc_dict = {}, {}, {}
    for i in range(y_score_local.shape[1]):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_bin_local[:, i], y_score_local[:, i])
        auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])
    return fpr_dict, tpr_dict, auc_dict

def compute_macro_roc(fpr_dict, tpr_dict, n_cls):
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(n_cls)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_cls):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= n_cls
    macro_auc_val = auc(all_fpr, mean_tpr)
    return all_fpr, mean_tpr, macro_auc_val

# =========================
# BASE ROC
# =========================
fpr_base, tpr_base, auc_base = compute_per_class_roc(y_bin, y_proba)
macro_fpr, macro_tpr, macro_auc = compute_macro_roc(fpr_base, tpr_base, n_classes)

# =========================
# BOOTSTRAP FOR MACRO STD BAND
# =========================
rng = np.random.RandomState(RANDOM_SEED)
macro_tprs_boot = []

for _ in range(N_BOOTSTRAPS):
    idx = rng.choice(len(y_true), size=len(y_true), replace=True)

    # need all classes present for valid multiclass bootstrap
    if len(np.unique(y_true[idx])) < n_classes:
        continue

    y_true_bs = y_true[idx]
    y_proba_bs = y_proba[idx]
    y_bin_bs = label_binarize(y_true_bs, classes=np.arange(n_classes))

    try:
        fpr_bs, tpr_bs, _ = compute_per_class_roc(y_bin_bs, y_proba_bs)
        macro_fpr_bs, macro_tpr_bs, _ = compute_macro_roc(fpr_bs, tpr_bs, n_classes)

        interp_macro_tpr = np.interp(macro_fpr, macro_fpr_bs, macro_tpr_bs)
        interp_macro_tpr[0] = 0.0
        interp_macro_tpr[-1] = 1.0
        macro_tprs_boot.append(interp_macro_tpr)
    except Exception:
        continue

macro_tprs_boot = np.array(macro_tprs_boot)

if len(macro_tprs_boot) > 0:
    macro_mean_tpr = macro_tprs_boot.mean(axis=0)
    macro_std_tpr = macro_tprs_boot.std(axis=0)
    macro_lower = np.maximum(macro_mean_tpr - macro_std_tpr, 0)
    macro_upper = np.minimum(macro_mean_tpr + macro_std_tpr, 1)
else:
    macro_mean_tpr = macro_tpr.copy()
    macro_lower = macro_tpr.copy()
    macro_upper = macro_tpr.copy()

# =========================
# PLOT
# =========================
plt.rcParams["font.family"] = FONT_FAMILY
plt.figure(figsize=FIGSIZE, dpi=DPI)

# per-class dotted ROC curves
for i in range(n_classes):
    plt.plot(
        fpr_base[i],
        tpr_base[i],
        linestyle=":",
        linewidth=CLASS_LINEWIDTH,
        color=CLASS_COLORS[i % len(CLASS_COLORS)],
        label=f"{class_names[i]} (AUC = {auc_base[i]:.3f})"
    )

# macro mean solid black line
plt.plot(
    macro_fpr,
    macro_tpr,
    color=MACRO_COLOR,
    linewidth=MACRO_LINEWIDTH,
    label=f"Macro-average (AUC = {macro_auc:.3f})"
)

# macro std-dev shaded band
plt.fill_between(
    macro_fpr,
    macro_lower,
    macro_upper,
    color=STD_COLOR,
    alpha=STD_ALPHA,
    label="Macro ±1 std"
)

# random baseline
plt.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)

plt.xlabel("False Positive Rate", fontsize=LABEL_SIZE)
plt.ylabel("True Positive Rate", fontsize=LABEL_SIZE)
plt.title("ROC Curve: Macro-Average and Per-Class", fontsize=TITLE_SIZE)

plt.xticks(fontsize=TICK_SIZE)
plt.yticks(fontsize=TICK_SIZE)
plt.legend(fontsize=LEGEND_SIZE, loc="lower right", frameon=True)
plt.grid(alpha=0.0)

plt.tight_layout()

if SAVE_PATH is not None:
    plt.savefig(SAVE_PATH, dpi=DPI, bbox_inches="tight")
    print(f"Saved → {SAVE_PATH}")

plt.show()